# Notebook 07: Model Calibration

**Goal**: Apply probability calibration to the best model to ensure predicted probabilities reflect true likelihood.

**Background**: Tree-based models often output extreme probabilities (near 0 or 1). Calibration adjusts these predictions so that, for example, among all predictions with 70% probability, approximately 70% are actually approved.

**Method**: Isotonic regression (non-parametric calibration, recommended for tree models)

## 1. Setup

In [1]:
import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.abspath(".."))

from src.preprocessing import load_preprocessed
from src.model_utils import load_model
from src.calibration import (
    calibrate_model,
    evaluate_calibration,
    compare_calibration,
    save_calibrated_model
)

print("✓ Imports successful")

✓ Imports successful


## 2. Load Data and Best Model

In [2]:
# Load preprocessed data
X_train, X_test, y_train, y_test, scaler = load_preprocessed("../outputs")

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Loaded  X_train=(16296, 14)  X_test=(2474, 14)
Training samples: 16296
Test samples: 2474


In [3]:
# Load best model from hyperparameter tuning
best_model = load_model("best_model", model_dir="../models")

print(f"\nBase model: {type(best_model).__name__}")
print(f"Parameters: {best_model.get_params()}")

Loaded ../models/best_model.pkl

Base model: RandomForestClassifier
Parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 9, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 20, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 150, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## 3. Evaluate Base Model Calibration

Check if base model needs calibration by examining:
- Calibration curve (should be diagonal)
- Probability distribution (should span 0-1)
- Calibration error (lower is better)

In [4]:
print("="*60)
print("EVALUATING BASE MODEL CALIBRATION")
print("="*60)

base_metrics = evaluate_calibration(
    best_model, X_test, y_test, 
    model_name="Base_Model",
    output_dir="../outputs"
)

print(f"\nCalibration Error: {base_metrics['calibration_error']:.4f}")
print(f"Probability Std: {base_metrics['prob_std']:.4f}")
print(f"Extreme Predictions: {base_metrics['extreme_prob_pct']:.1f}%")

EVALUATING BASE MODEL CALIBRATION

Evaluating calibration for Base_Model
  Calibration error: 0.2145 (lower is better)
  Prob mean: 0.387, std: 0.188
  Extreme predictions (<0.1 or >0.9): 0.2%
  ✓ Calibration curve saved to ../outputs/calibration_curve_Base_Model.png

Calibration Error: 0.2145
Probability Std: 0.1877
Extreme Predictions: 0.2%


## 4. Apply Calibration

Use `CalibratedClassifierCV` with isotonic regression and 5-fold CV.

In [5]:
calibrated_model = calibrate_model(
    best_model, X_train, y_train,
    method='isotonic',
    cv=5
)


Calibrating model with method='isotonic', cv=5
  Base model: RandomForestClassifier


  ✓ Calibration complete


## 5. Evaluate Calibrated Model

In [6]:
print("\n" + "="*60)
print("EVALUATING CALIBRATED MODEL")
print("="*60)

cal_metrics = evaluate_calibration(
    calibrated_model, X_test, y_test,
    model_name="Calibrated_Model",
    output_dir="../outputs"
)

print(f"\nCalibration Error: {cal_metrics['calibration_error']:.4f}")
print(f"Probability Std: {cal_metrics['prob_std']:.4f}")
print(f"Extreme Predictions: {cal_metrics['extreme_prob_pct']:.1f}%")


EVALUATING CALIBRATED MODEL

Evaluating calibration for Calibrated_Model
  Calibration error: 0.2696 (lower is better)
  Prob mean: 0.317, std: 0.293
  Extreme predictions (<0.1 or >0.9): 38.3%


  ✓ Calibration curve saved to ../outputs/calibration_curve_Calibrated_Model.png

Calibration Error: 0.2696
Probability Std: 0.2926
Extreme Predictions: 38.3%


## 6. Compare Base vs Calibrated

In [7]:
comparison = compare_calibration(
    best_model, calibrated_model,
    X_test, y_test,
    output_dir="../outputs"
)


CALIBRATION COMPARISON: Base vs Calibrated

Evaluating calibration for Base Model
  Calibration error: 0.2145 (lower is better)
  Prob mean: 0.387, std: 0.188
  Extreme predictions (<0.1 or >0.9): 0.2%
  ✓ Calibration curve saved to ../outputs/calibration_curve_Base_Model.png

Evaluating calibration for Calibrated Model


  Calibration error: 0.2696 (lower is better)
  Prob mean: 0.317, std: 0.293
  Extreme predictions (<0.1 or >0.9): 38.3%
  ✓ Calibration curve saved to ../outputs/calibration_curve_Calibrated_Model.png

------------------------------------------------------------
COMPARISON SUMMARY:
  Calibration error:
    Base:       0.2145
    Calibrated: 0.2696
    Improvement: -0.0551

  Probability std:
    Base:       0.1877
    Calibrated: 0.2926

  Extreme predictions (%):
    Base:       0.2%
    Calibrated: 38.3%



## 7. Save Calibrated Model

Save to `models/best_model_calibrated.pkl` for use in the API.

In [8]:
save_calibrated_model(
    calibrated_model,
    model_name="best_model_calibrated",
    model_dir="../models"
)

print("\n" + "="*60)
print("✓ CALIBRATION COMPLETE")
print("="*60)
print("\nNext steps:")
print("  1. Restart the backend API (it will auto-load calibrated model)")
print("  2. Test predictions with borderline cases (credit_score=650, DTI=0.40)")
print("  3. Verify probabilities are no longer extreme (0 or 1)")
print("  4. Run notebooks 04-06 for SHAP/DiCE/Fairness analysis")

✓ Saved calibrated model to ../models/best_model_calibrated.pkl

✓ CALIBRATION COMPLETE

Next steps:
  1. Restart the backend API (it will auto-load calibrated model)
  2. Test predictions with borderline cases (credit_score=650, DTI=0.40)
  3. Verify probabilities are no longer extreme (0 or 1)
  4. Run notebooks 04-06 for SHAP/DiCE/Fairness analysis
